# NES-VMC H2/6-31G K=6 —— 交互式训练可视化版本

基于 `H2_631G_8Qbit_K6.ipynb`（复参数 NES-VMC + 双侧列规范 + gauge reset），新增：

1. **训练实时动画**：每步刷新 6 联图（能级 / Loss / 梯度范数 / logPsi / cond(Ψ) / E_L trace），使用 `ipympl` 后端，可在 Notebook 前端直接缩放、平移、悬停读数。
2. **训练后交互探索**：`ipywidgets` 控件（能级下拉 + 起始步滑块）浏览各能级误差随训练步的变化。

> 无 `ipympl` 前端环境（如纯终端执行）自动回退为训练结束后一次性静态绘图。
> 超参默认与原 notebook 一致（N_ITER=300, N_CHAINS=16）；可用环境变量 `NES_N_ITER` / `NES_N_CHAINS` 覆盖以便快速冒烟测试。

In [ ]:
import logging
import os
import time
import h5py
import jax
import jax.numpy as jnp
import flax.nnx as nnx
import netket as nk
import optax
from scipy.sparse.linalg import eigsh
import pickle
import numpy as np
from NES_VMC_V1 import (
    NESTotalAnsatz,
    NESTotalAnsatz_stable,
    create_single_machine_gauge_fixed,
    Ham_Psi_scaled,
    NESFermionHopRule,
    ravel_pytree,
)
from NES_VMC_tool import create_gauge_reset_total_machines, NES_loss_energy_stable_gauge, \
    nes_vmc_gradient_stable_gauge, make_grad_fn_gauge, make_qgt_fn_gauge, make_gauge_fn
from H2_6_31G import SINGLE_SIZE, ha, hi_ext, ext_edges, K, hi, E_fcis, Hatree_Fock

K = 6
jax.config.update("jax_enable_x64", False)  # 与原 notebook 保持一致（默认 float32）
print("jax x64:", jax.config.jax_enable_x64)
print("netket:", nk.__version__, "| jax:", jax.__version__)

In [ ]:
# ====================== 日志配置 ======================
time_str = time.strftime("%y-%m-%d-%H-%M")   # 输出文件名统一时间前缀 YY-DD-HH-MM
logger = logging.getLogger("NES_VMC_K6_interactive")
logger.setLevel(logging.INFO)
logger.propagate = False
logger.handlers.clear()
simple_formatter = logging.Formatter("%(asctime)s %(message)s", datefmt="%y-%d-%H-%M")

os.makedirs("./日志", exist_ok=True)
log_path = f"./日志/{time_str}_nes_vmc_K6_H2_6-31G_interactive.log"
file_handler = logging.FileHandler(log_path, mode="w", encoding="utf-8")
file_handler.setFormatter(simple_formatter)
logger.addHandler(file_handler)

console_handler = logging.StreamHandler()
console_handler.setFormatter(simple_formatter)
logger.addHandler(console_handler)

# ====================== 超参配置 ======================
N_CHAINS = int(os.environ.get("NES_N_CHAINS", "16"))
N_SAMPLES_PER_CHAIN = 200
SWEEP_SIZE = 20
N_ITER = int(os.environ.get("NES_N_ITER", "300"))
Natural_Grad = True
clip_norm = 20.0
per_param_clip = 0.5
lr = 0.1
qgt_diag_shift = 0.1
RESET_PERIOD = 10
SAVE_INTERVAL = 20
PLOT_EVERY = 1               # 每多少步刷新一次交互图
HISTORY_FILE = f"./data/{time_str}_history_natural_gradient_H2_molecule_K6_interactive.pkl"
os.makedirs("./data", exist_ok=True)

# ====================== 模型与采样器 ======================
total_ansatz = NESTotalAnsatz_stable(
    n_spin_orbitals=SINGLE_SIZE,
    n_states=K,
    hidden_dim=SINGLE_SIZE + K,
    rngs=nnx.Rngs(11),
)

g_current = jnp.zeros(K, dtype=jnp.complex64)   # 全局列规范 g，非训练参数，动态传入 jit

(
    total_machine,
    total_matrix_machine,
    total_max_machine,
    total_matrix_machine_raw,
    total_graphdef,
    total_params,
) = create_gauge_reset_total_machines(total_ansatz, Hatree_Fock)

single_machine_list = [
    create_single_machine_gauge_fixed(ansatz, Hatree_Fock)[0]
    for ansatz in total_ansatz.single_ansatz_list
]

grad_fn = make_grad_fn_gauge(
    ha, total_matrix_machine, total_max_machine, total_machine, single_machine_list,
)
qgt_fn = make_qgt_fn_gauge(total_machine)
gauge_fn, col_mean_fn = make_gauge_fn(total_ansatz, Hatree_Fock)

nes_rule = NESFermionHopRule(edges=ext_edges, K=K, single_size=SINGLE_SIZE)
nes_sampler = nk.sampler.MetropolisSampler(
    hilbert=hi_ext, rule=nes_rule, n_chains=N_CHAINS, sweep_size=SWEEP_SIZE,
)

exact_eigvals = E_fcis
optimizer = optax.chain(optax.clip_by_global_norm(clip_norm), optax.sgd(learning_rate=lr))
opt_state = optimizer.init(total_params)
sampler_rng = jax.random.PRNGKey(21)


def sample_machine(params, sigma):
    """采样只需要 |Ψ|² 的转移率，全局列规范 g 是常数平移，不影响 ratio。"""
    return total_machine(params, sigma, g_current)


sampler_state = nes_sampler.init_state(sample_machine, total_params, sampler_rng)

# ====================== 历史记录 ======================
loss_history = []
steps_history = []
logpsi_mean_history = []
logpsi_min_history = []
logpsi_max_history = []
grad_norm_raw_history = []
grad_norm_per_param_history = []
grad_norm_nat_history = []
E_L_real_history = []
E_L_imag_history = []
cond_history = []
params_history = []
Energy_levels_history = []
samples_history = []

logger.info("\n" + "=" * 60)
logger.info("开始多链 NES-VMC 训练（交互式可视化版）| K=6 | 双侧一致列规范 + 周期性 gauge reset")
logger.info(f"N_CHAINS={N_CHAINS}, N_SAMPLES_PER_CHAIN={N_SAMPLES_PER_CHAIN}, SWEEP_SIZE={SWEEP_SIZE}, "
            f"N_ITER={N_ITER}, lr={lr}, clip_norm={clip_norm}, per_param_clip={per_param_clip}, "
            f"qgt_diag_shift={qgt_diag_shift}, RESET_PERIOD={RESET_PERIOD}")
logger.info(f"理论 Loss 上限：{sum(exact_eigvals[0:K]):.8f}")
logger.info("=" * 60)

In [ ]:
# ====================== 交互式绘图设置（ipympl 实时动画）======================
import matplotlib
import matplotlib.pyplot as plt

INTERACTIVE = False
try:
    get_ipython().run_line_magic("matplotlib", "ipympl")
    import ipympl  # noqa: F401
    INTERACTIVE = True
    print("已启用 ipympl 后端：训练过程中图会实时刷新，可缩放/平移/悬停读数。")
except Exception as e:
    print(f"ipympl 不可用（{type(e).__name__}: {e}），回退为训练结束后一次性静态绘图。")

fig, axes = plt.subplots(3, 2, figsize=(13, 11))
ax_e, ax_loss, ax_grad, ax_psi, ax_cond, ax_el = axes.ravel()


def refresh(step_now=None):
    """清空并重画 6 联图（数据量 <= N_ITER，开销可接受）。"""
    steps_arr = np.asarray(steps_history)
    E_arr = np.asarray(Energy_levels_history)          # (n, K)
    title = f"NES-VMC H2/6-31G K=6 (interactive)" + (f"  step={step_now}" if step_now is not None else "")

    ax_e.clear(); ax_e.plot(steps_arr, E_arr, lw=1.2)
    for i in range(K):
        ax_e.axhline(float(exact_eigvals[i]), color="gray", ls="--", alpha=0.4, lw=0.8)
    ax_e.set_ylabel("Eigenvalues (Ha)"); ax_e.set_xlabel("step")
    ax_e.set_ylim(-1.2, 0.3); ax_e.set_title(title)

    ax_loss.clear(); ax_loss.plot(steps_arr, loss_history, color="tab:orange", lw=1.2)
    ax_loss.set_ylabel("Loss (Ha)"); ax_loss.set_xlabel("step"); ax_loss.set_title("Loss = Re tr(E_L)")

    ax_grad.clear()
    ax_grad.semilogy(steps_arr, grad_norm_raw_history, label="raw", lw=1.2)
    ax_grad.semilogy(steps_arr, grad_norm_per_param_history, label="per_param", lw=1.2)
    ax_grad.semilogy(steps_arr, grad_norm_nat_history, label="natural", lw=1.2)
    ax_grad.axhline(clip_norm, color="red", ls=":", lw=0.8, label="clip_norm")
    ax_grad.set_ylabel("Grad norm (log)"); ax_grad.set_xlabel("step"); ax_grad.legend(fontsize=8)
    ax_grad.set_title("Gradient norms")

    ax_psi.clear()
    ax_psi.plot(steps_arr, logpsi_mean_history, label="mean", lw=1.2)
    ax_psi.plot(steps_arr, logpsi_min_history, label="min", lw=1.0, alpha=0.7)
    ax_psi.plot(steps_arr, logpsi_max_history, label="max", lw=1.0, alpha=0.7)
    ax_psi.set_ylabel("Re logPsi"); ax_psi.set_xlabel("step"); ax_psi.legend(fontsize=8)
    ax_psi.set_title("logPsi statistics")

    ax_cond.clear(); ax_cond.semilogy(steps_arr, cond_history, color="tab:purple", lw=1.2)
    ax_cond.set_ylabel("cond(psi_mat)"); ax_cond.set_xlabel("step"); ax_cond.set_title("Psi matrix condition number")

    ax_el.clear()
    ax_el.plot(steps_arr, E_L_real_history, label="Re tr(E_L)", lw=1.2)
    ax_el.plot(steps_arr, E_L_imag_history, label="Im tr(E_L)", lw=1.2)
    ax_el.set_ylabel("E_L trace"); ax_el.set_xlabel("step"); ax_el.legend(fontsize=8)
    ax_el.set_title("E_L trace (imag should stay ~0)")

    fig.tight_layout()
    if INTERACTIVE:
        try:
            fig.canvas.draw_idle()
        except Exception:
            pass  # 无头环境刷新失败不影响训练


if INTERACTIVE:
    from IPython.display import display
    display(fig)
    refresh(0)
else:
    print(f"训练配置：N_ITER={N_ITER}, N_CHAINS={N_CHAINS}（结束后自动绘图）")

In [ ]:
# ====================== 训练循环（每步实时刷新交互图）======================
start_time = time.time()

for step in range(N_ITER):
    params_history.append(total_params)
    # 1. 采样
    samples_raw, sampler_state = nes_sampler.sample(
        machine=sample_machine, parameters=total_params,
        state=sampler_state, chain_length=N_SAMPLES_PER_CHAIN,
    )
    samples = samples_raw.reshape(-1, hi_ext.size)
    x_batch = samples.reshape(-1, K, SINGLE_SIZE)

    # 2. 梯度与损失
    grad_raw, loss_mean, E_L_mean, aux = grad_fn(total_params, x_batch, g_current)

    grad_raw_flat, unravel_fn = ravel_pytree(grad_raw)
    grad_norm_raw = jnp.linalg.norm(grad_raw_flat)
    n_params = grad_raw_flat.size
    grad_norm_per_param = grad_norm_raw / jnp.sqrt(n_params)

    if float(grad_norm_per_param) > per_param_clip:
        grad_scale = per_param_clip / float(grad_norm_per_param)
        grad_raw_flat = grad_raw_flat * grad_scale
        grad_raw = unravel_fn(grad_raw_flat)
        grad_norm_raw = jnp.linalg.norm(grad_raw_flat)
        grad_norm_per_param = grad_norm_raw / jnp.sqrt(n_params)
        logger.info(f"【Step {step}】per-param 梯度范数超限，已裁剪：scale={grad_scale:.4f} | 裁剪后 per_param={float(grad_norm_per_param):.4f}")

    grad_update = grad_raw

    has_nan = bool(jnp.any(jnp.isnan(grad_raw_flat)))
    if has_nan or grad_norm_raw > 5000.0:
        logger.warning(f"【Step {step} 告警】梯度异常！nan={has_nan}, raw_grad_norm={grad_norm_raw:.2f}")

    # 3. QGT 自然梯度
    if Natural_Grad:
        qgt_reg_mat = qgt_fn(total_params, x_batch, qgt_diag_shift, g_current)
        ng_flat = jnp.linalg.solve(qgt_reg_mat, grad_raw_flat)
        grad_update = unravel_fn(ng_flat)
        grad_norm_natural = jnp.linalg.norm(ng_flat)
    else:
        grad_norm_natural = grad_norm_raw

    # 4. 优化器更新
    updates, opt_state = optimizer.update(grad_update, opt_state, total_params)
    total_params = optax.apply_updates(total_params, updates)
    grad_norm_clipped = min(float(grad_norm_natural), clip_norm)

    # 5. 能量监控：E_L 矩阵本征值
    eig_vals, _ = jnp.linalg.eig(E_L_mean)
    eig_vals = eig_vals[jnp.argsort(eig_vals.real)]

    # 6. 波函数与 Ψ 矩阵条件数监控
    log_Psi_batch = total_machine(total_params, x_batch, g_current)
    x_single = x_batch[0:1, ...]
    psi_mat = total_matrix_machine(total_params, x_single, g_current)[0]
    psi_cond = jnp.linalg.cond(psi_mat)

    gauge_now = float(jnp.real(gauge_fn(total_params, x_batch)))
    g_abs = float(jnp.linalg.norm(g_current))

    # 7. 日志
    logger.info(f"[Step {step:3d}] logΨ: mean={log_Psi_batch.mean():.3f} | min={log_Psi_batch.min():.3f} | max={log_Psi_batch.max():.3f}")
    logger.info(f"梯度监控 | raw={grad_norm_raw:.4f} | per_param={grad_norm_per_param:.4f} | natural={grad_norm_natural:.4f} | clipped={grad_norm_clipped:.4f}(上限{clip_norm})")
    logger.info(f"原始规范坐标 G = {gauge_now:+.4f} | 累计|g| = {g_abs:.4f}")
    logger.info(f"Ψ矩阵条件数 cond(Ψ) = {psi_cond:.2e}")
    logger.info(
        f"Loss={loss_mean:.6f} | E0={eig_vals[0]:.8f} | E1={eig_vals[1]:.8f}"
        f"|E2={eig_vals[2]:.8f}|E3={eig_vals[3]:.8f}|E4={eig_vals[4]:.8f}|E5={eig_vals[5]:.8f}"
    )
    logger.info("#-----------------------------------------#")

    # 记录历史
    loss_history.append(float(jnp.real(loss_mean)))
    steps_history.append(step)
    logpsi_mean_history.append(float(jnp.real(log_Psi_batch.mean())))
    logpsi_min_history.append(float(jnp.real(log_Psi_batch.min())))
    logpsi_max_history.append(float(jnp.real(log_Psi_batch.max())))
    grad_norm_raw_history.append(float(grad_norm_raw))
    grad_norm_nat_history.append(float(grad_norm_natural))
    grad_norm_per_param_history.append(float(grad_norm_per_param))
    E_L_real_history.append(float(jnp.real(jnp.trace(E_L_mean))))
    E_L_imag_history.append(float(jnp.imag(jnp.trace(E_L_mean))))
    cond_history.append(float(psi_cond))
    Energy_levels_history.append(eig_vals[:K])
    samples_history.append(samples)

    # >>> 交互式实时刷新 <<<
    if INTERACTIVE and (step % PLOT_EVERY == 0):
        refresh(step)

    # 周期性保存 pickle
    if (step + 1) % SAVE_INTERVAL == 0:
        history = {
            "steps": [int(s) for s in steps_history],
            "logpsi_mean": logpsi_mean_history, "logpsi_min": logpsi_min_history,
            "logpsi_max": logpsi_max_history, "grad_norm_raw": grad_norm_raw_history,
            "grad_norm_natural": grad_norm_nat_history, "grad_norm_per_param": grad_norm_per_param_history,
            "E_L_real": E_L_real_history, "E_L_imag": E_L_imag_history,
            "cond": cond_history, "loss": loss_history,
            "save_interval": SAVE_INTERVAL, "Energy_levels": Energy_levels_history,
            "params": params_history,
        }
        with open(HISTORY_FILE, "wb") as f:
            pickle.dump(history, f)
        logger.info(f"[保存] Step {step} → pickle 文件已更新 ({len(steps_history)} 条记录)")

    # 周期性 gauge reset
    if (step + 1) % RESET_PERIOD == 0:
        new_col_mean = col_mean_fn(total_params, x_batch)
        g_current = new_col_mean
        absorbed = float(jnp.sum(jnp.real(new_col_mean)))
        logger.info(f"[GaugeReset@{step + 1}] 吸收 Δg = {absorbed:+.4f} | 新|g| = {float(jnp.linalg.norm(g_current)):.4f}")

end_time = time.time()
logger.info(f"训练耗时：{end_time - start_time:.2f} 秒")
logger.info("\n" + "=" * 60 + "\n训练完成!\n" + "=" * 60)

# 最终 pickle 写入
history = {
    "steps": [int(s) for s in steps_history],
    "logpsi_mean": logpsi_mean_history, "logpsi_min": logpsi_min_history,
    "logpsi_max": logpsi_max_history, "grad_norm_raw": grad_norm_raw_history,
    "grad_norm_natural": grad_norm_nat_history, "grad_norm_per_param": grad_norm_per_param_history,
    "E_L_real": E_L_real_history, "E_L_imag": E_L_imag_history,
    "cond": cond_history, "loss": loss_history,
    "save_interval": SAVE_INTERVAL, "Energy_levels": Energy_levels_history,
    "params": params_history, "samples": samples_history,
}
with open(HISTORY_FILE, "wb") as f:
    pickle.dump(history, f)
logger.info(f"[保存] 训练结束，pickle 最终写入 {len(steps_history)} 条记录 → {HISTORY_FILE}")

# 训练结束：无论哪种模式都绘制最终图（ipympl 下 plt.show() 无害，部分前端需要它生成快照）
refresh(N_ITER - 1 if N_ITER > 0 else 0)
try:
    plt.show()
except Exception:
    pass

In [ ]:
# ====================== 训练后交互探索：能级误差随步数变化 ======================
import ipywidgets as widgets
from IPython.display import display

E_arr = np.asarray(Energy_levels_history)                       # (n_steps, K)
err_mHa = np.abs(E_arr - np.asarray(exact_eigvals[:K])[None, :]) * 1000.0   # 误差 (mHa)

level_dd = widgets.Dropdown(options=[f"E{i}" for i in range(K)], value="E0", description="能级：")
start_sl = widgets.IntSlider(min=0, max=max(len(steps_history) - 2, 0), value=0,
                             continuous_update=False, description="起始步：")
log_cb = widgets.Checkbox(value=True, description="对数坐标")
out = widgets.Output()


def on_change(_):
    with out:
        out.clear_output(wait=True)
        i = int(level_dd.value[1:])
        s = int(start_sl.value)
        f2, a2 = plt.subplots(figsize=(10, 3.5))
        a2.plot(np.asarray(steps_history)[s:], err_mHa[s:, i], lw=1.4, color="tab:blue")
        a2.axhline(1.6, color="red", ls="--", lw=1.0, label="chemical accuracy ~1.6 mHa")
        if log_cb.value:
            a2.set_yscale("log")
        a2.set_xlabel("step")
        a2.set_ylabel(f"|E{i} - E{i}(FCI)| (mHa)")
        a2.set_title(f"E{i} error vs step (from step {s})")
        a2.legend(fontsize=8)
        f2.tight_layout()
        plt.show()


widgets.interactive_output(on_change, {"lvl": level_dd, "st": start_sl, "lg": log_cb})
display(widgets.VBox([widgets.HBox([level_dd, start_sl, log_cb]), out]))
on_change(None)